# Verifying autoregressive B-RASP programs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TuLLI-AI/symprog/blob/main/notebooks/autoregressive_brasp.ipynb)

In [`b_rasp.ipynb`](b_rasp.ipynb), a B-RASP program *recognizes* strings. Here it *generates* them, one symbol at a time, like a language model. At each step, a few B-RASP formulas are evaluated on the string so far, and their truth values pick the distribution of the next symbol (or of `eos`, which stops generation).

We ask a verification question:

> Given a **prompt** $u$, a **cap** $K$ on the number of generated symbols, and a **property** $L$ (a regular language, given by a DFA), what is the probability that the model generates a continuation $v$, ending with `eos`, such that $uv \in L$?

$$\Pr{}^K_A(L_u \mid u) \;=\; \sum_{\substack{v:\ uv \in L \\ |v| \le K}} \;\Big(\prod_{j=1}^{|v|} \Pr_A(v_j \mid u v_{<j})\Big)\cdot \Pr_A(\textsc{eos} \mid uv)$$

Asking whether this equals (or exceeds) a threshold $\lambda$ is #P-hard (resp. PP-hard), so there is no shortcut in general. But the whole computation is **Boolean** for a fixed $K$, so it can be written as a **weighted model count**: one satisfying assignment per generation path, with the path's probability as its weight. `symprog` then compiles it into a circuit and counts.

The notebook follows the draft *Verifying (Probabilistic) Autoregressive UHAT*:

1. the model: B-RASP features and a readout,
2. the property: a DFA,
3. the answer by brute force (enumerating every continuation),
4. the same answer as a weighted model count, built step by step,
5. what else the circuit gives us.

## Setup

In [1]:
# %pip install -q symprog

## use the installed symprog, or else the version in this repository
try:
    import symprog
except ImportError:
    import pathlib, sys
    repo = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "symprog" / "__init__.py").exists()), None)
    if repo is None:
        raise ImportError("symprog is not installed: pip install symprog, or run this notebook from the symprog repository")
    sys.path.insert(0, str(repo))
    import symprog

import itertools
import time
from fractions import Fraction

from symprog import Solver

# 1. The model

## B-RASP, as in `b_rasp.ipynb`

The same small embedding as in [`b_rasp.ipynb`](b_rasp.ipynb), with two differences that match the draft:

- attention is always **strict-past rightmost**: $\blacktriangleright_j [j < i, S(i,j)]\; V(i,j) : D(i)$ finds the rightmost earlier position $j$ where the score $S$ holds, and returns $V(i,j)$, or the default $D(i)$ if there is none;
- a positional predicate $\text{MOD}^r_m(i)$, true when $i \equiv r \pmod m$ (positions are numbered from 1, prompt included).

The Boolean helpers work on Python Booleans and on formula strings, and simplify constants away:

In [2]:
def And(*xs):
    if any(x is False for x in xs):
        return False
    xs = [x for x in xs if x is not True]
    return True if not xs else xs[0] if len(xs) == 1 else f"And({', '.join(xs)})"

def Or(*xs):
    if any(x is True for x in xs):
        return True
    xs = [x for x in xs if x is not False]
    return False if not xs else xs[0] if len(xs) == 1 else f"Or({', '.join(xs)})"

def Not(x):
    return (not x) if isinstance(x, bool) else f"Not({x})"

def Implies(x, y):
    return Or(Not(x), y)

def Iff(x, y):
    if isinstance(x, bool):
        return y if x else Not(y)
    if isinstance(y, bool):
        return x if y else Not(x)
    return f"Biconditional({x}, {y})"

def ITE(condition, then, otherwise):
    return Or(And(condition, then), And(Not(condition), otherwise))

In [3]:
class Vector:
    """A Boolean vector, indexed by positions 1..n."""
    def __init__(self, values):
        self.values = values

    def __getitem__(self, i):
        return self.values[i - 1]

class Builder:
    """The B-RASP operations. Subclasses decide how a new vector is stored (`define`)."""

    def position(self, name, f):
        """P(i) := f(i), a Boolean combination of vectors at position i."""
        return self.define(name, [f(i) for i in self.positions])

    def rightmost(self, name, score, value, default):
        """P(i) := >_j [j < i, score(i, j)] value(i, j) : default(i)."""
        exprs = []
        for i in self.positions:
            expr = default(i)
            for j in range(1, i):  ## the rightmost j ends up as the outermost test
                expr = ITE(score(i, j), value(i, j), expr)
            exprs.append(expr)
        return self.define(name, exprs)

    def mod(self, m, r):
        """MOD^r_m(i): a constant at each position."""
        return Vector([i % m == r for i in self.positions])

class Run(Builder):
    """Runs a program on one string, with Python Booleans."""
    def __init__(self, alphabet, word):
        self.positions = range(1, len(word) + 1)
        self.Q = {a: Vector([c == a for c in word]) for a in alphabet}

    def define(self, name, values):
        return Vector(values)

## Warm-up: B-RASP on `abba`

Before generating anything, here is how a B-RASP program computes, on the word `abba`. Every value is a **Boolean vector**, with one bit per position. There are three kinds of operations:

1. **symbol predicates** $Q_\sigma$: the starting vectors, true where the word holds $\sigma$ (the token embedding);
2. **position-wise** operations: $\wedge$, $\vee$, $\neg$ of existing vectors, at the **same** position (the MLP);
3. **attention**: the only way information moves between positions.

`show` prints a vector as a row of bits:

In [4]:
p = Run(["a", "b"], "abba")

def show(name, vector):
    print(f"{name:10}", " ".join("1" if vector[i] else "0" for i in p.positions))

print(f"{'position':10}", " ".join(str(i) for i in p.positions))
print(f"{'word':10}", " ".join("abba"))
show("Q_a", p.Q["a"])
show("Q_b", p.Q["b"])

position   1 2 3 4
word       a b b a
Q_a        1 0 0 1
Q_b        0 1 1 0


#### Attention: "is the previous symbol an $a$?"

$$\textit{prev\_a}(i) := \blacktriangleright_j [j < i,\ \top]\; Q_a(j) : 0$$

At position $i$, look at the earlier positions $j < i$ where the **score** holds, take the **rightmost** one, and return the **value** there, or the **default** if there is none. Here the score $\top$ holds everywhere, so the rightmost earlier position is $i - 1$, and this reads $Q_a$ one position back. Position 1 has no earlier position, so it gets the default 0.

In [5]:
prev_a = p.rightmost("prev_a", lambda i, j: True, lambda i, j: p.Q["a"][j], lambda i: False)
show("prev_a", prev_a)

prev_a     0 1 0 0


#### The score filters: "has an $a$ appeared before?", then a position-wise operation

$$\textit{seen\_a}(i) := \blacktriangleright_j [j < i,\ Q_a(j)]\; 1 : 0 \qquad\qquad \textit{b\_after\_a}(i) := Q_b(i) \wedge \textit{seen\_a}(i)$$

Now only earlier positions holding an $a$ match the score. The value is the constant 1, so the attention asks whether there is any match. The second operation is position-wise: it combines two vectors column by column.

In [6]:
seen_a = p.rightmost("seen_a", lambda i, j: p.Q["a"][j], lambda i, j: True, lambda i: False)
b_after_a = p.position("b_after_a", lambda i: And(p.Q["b"][i], seen_a[i]))
show("seen_a", seen_a)
show("b_after_a", b_after_a)

seen_a     0 1 1 1
b_after_a  0 1 1 0


#### Score and value do different jobs: "was the most recent earlier $b$ preceded by an $a$?"

$$x(i) := \blacktriangleright_j [j < i,\ Q_b(j)]\; \textit{prev\_a}(j) : 0$$

The **score** picks *which* position to look at (the most recent earlier $b$), and the **value** says *what to read* there (another vector, `prev_a`, computed by an earlier attention: this is how layers compose).

At position 4, both positions 2 and 3 hold a $b$. The rightmost, 3, wins, and $\textit{prev\_a}(3) = 0$. With leftmost attention, position 4 would read $\textit{prev\_a}(2) = 1$ instead.

In [7]:
x = p.rightmost("x", lambda i, j: p.Q["b"][j], lambda i, j: prev_a[j], lambda i: False)
show("prev_a", prev_a)
show("Q_b", p.Q["b"])
show("x", x)

prev_a     0 1 0 0
Q_b        0 1 1 0
x          0 0 1 0


#### B-RASP and transformers

| B-RASP | transformer |
|---|---|
| a vector (one bit per position) | one coordinate of the hidden state at each token |
| $Q_\sigma$ | token embedding |
| $\text{MOD}^r_m$ | positional encoding |
| position-wise $\wedge, \vee, \neg$ | MLP |
| score $S(i, j)$ | attention logits, restricted to 0 or 1 |
| $j < i$ | causal mask (strict) |
| rightmost match | hard attention, ties broken to the right ("UHAT") |
| default $D(i)$ | nothing to attend to |

The score and value may mention both $i$ and $j$, e.g., $Q_a(i) \wedge Q_a(j)$ for "positions with the same symbol as me".

**Recognizing vs. generating.** A recognizer (as in `b_rasp.ipynb`) reads one vector at the **last** position: $x(4) = 0$ rejects `abba`. A generator reads a few vectors at the last position, here $(\textit{prev\_a}(4), \textit{seen\_a}(4), x(4)) = (0, 1, 0)$, and a readout table maps these bits to a distribution over the next symbol or `eos`. The sampled symbol is appended, every vector gets one more position, and the process repeats. That is the model below.

## An autoregressor $A = (\Phi, r)$

A toy language model over $\Sigma = \{a, b\}$. Its **features** $\Phi$ are three B-RASP vectors, read at the last position of the string so far:

- `cur_a`: the last symbol is $a$, i.e., $Q_a(i)$;
- `prev_a`: the symbol before it is $a$, i.e., $\blacktriangleright_j [j < i, \top]\; Q_a(j) : \bot$;
- `even`: the last position is even, i.e., $\text{MOD}^0_2(i)$.

The **readout** $r$ maps the three truth values to a distribution over $\{a, b, \textsc{eos}\}$. The model likes repeating $a$, prefers to stop after an $a$ at an even position, and never stops right after a $b$. For the empty prompt there is no last position, so the first symbol has its own distribution `FIRST`.

In [8]:
ALPHABET = ["a", "b"]

def features(p):
    cur_a = p.Q["a"]
    prev_a = p.rightmost("prev_a", lambda i, j: True, lambda i, j: p.Q["a"][j], lambda i: False)
    even = p.mod(2, 0)
    return [cur_a, prev_a, even]

def readout(v):
    cur_a, prev_a, even = v
    if not cur_a:
        return {"a": 0.25, "b": 0.75, "eos": 0}
    if even:
        return {"a": 0.25, "b": 0.25, "eos": 0.5}
    if prev_a:
        return {"a": 0.5, "b": 0.25, "eos": 0.25}
    return {"a": 0.75, "b": 0.25, "eos": 0}

FIRST = {"a": 0.5, "b": 0.5, "eos": 0}

The next-symbol distribution after a prefix: run the features on the prefix, and read them at its last position. Since attention only looks at earlier positions, the features at position $i$ depend only on the first $i$ symbols.

In [9]:
def next_distribution(prefix):
    if prefix == "":
        return FIRST
    phi = features(Run(ALPHABET, prefix))
    return readout(tuple(f[len(prefix)] for f in phi))

word = "abaab"
for i in range(len(word) + 1):
    print(f"{word[:i]!r:8} -> {next_distribution(word[:i])}")

''       -> {'a': 0.5, 'b': 0.5, 'eos': 0}
'a'      -> {'a': 0.75, 'b': 0.25, 'eos': 0}
'ab'     -> {'a': 0.25, 'b': 0.75, 'eos': 0}
'aba'    -> {'a': 0.75, 'b': 0.25, 'eos': 0}
'abaa'   -> {'a': 0.25, 'b': 0.25, 'eos': 0.5}
'abaab'  -> {'a': 0.25, 'b': 0.75, 'eos': 0}


The probability of generating a continuation $v$ after the prompt $u$, and then stopping, is the product of the next-symbol probabilities along the way, times the probability of `eos` at the end:

In [10]:
def path_probability(prompt, continuation):
    prob = Fraction(1)
    for j in range(len(continuation)):
        prob *= Fraction(next_distribution(prompt + continuation[:j])[continuation[j]])
    return prob * Fraction(next_distribution(prompt + continuation)["eos"])

for v in ["", "a", "aa", "aaa", "aaaa"]:
    print(f"Pr({v!r:6} | 'ab') = {path_probability('ab', v)}")

Pr(''     | 'ab') = 0
Pr('a'    | 'ab') = 0
Pr('aa'   | 'ab') = 3/32
Pr('aaa'  | 'ab') = 3/256
Pr('aaaa' | 'ab') = 3/256


# 2. The property

The property is a DFA over $\Sigma$. It reads the prompt and the continuation, but not `eos`. Here: **no two consecutive $b$'s**, with state 0 (last symbol is not $b$), 1 (last symbol is $b$) and 2 (seen $bb$, rejecting forever).

In [11]:
STATES = [0, 1, 2]
DELTA = {(0, "a"): 0, (0, "b"): 1,
         (1, "a"): 0, (1, "b"): 2,
         (2, "a"): 2, (2, "b"): 2}
START, ACCEPTING = 0, {0, 1}

def dfa_state(word):
    state = START
    for c in word:
        state = DELTA[state, c]
    return state

for w in ["abab", "abba"]:
    print(w, "accepted" if dfa_state(w) in ACCEPTING else "rejected")

abab accepted
abba rejected


# 3. The answer by brute force

Enumerate every continuation of length at most $K$, and add up the probabilities of those that satisfy the property. This takes $|\Sigma|^{K}$ runs, so it only works for small $K$, but it is the definition, and we use it to check the encoding.

In [12]:
def brute_force(prompt, K):
    total = Fraction(0)
    for m in range(K + 1):
        for v in map("".join, itertools.product(ALPHABET, repeat=m)):
            if dfa_state(prompt + v) in ACCEPTING:
                total += path_probability(prompt, v)
    return total

prompt, K = "ab", 3
print(f"Pr^{K}(no bb | {prompt!r}) =", brute_force(prompt, K))

Pr^3(no bb | 'ab') = 27/256


# 4. The answer as a weighted model count

We now build a set of Boolean formulas whose satisfying assignments are exactly the generation paths that stop within the cap and satisfy the property, with weight = path probability. The pieces are:

| piece | variables | what it says |
|---|---|---|
| (a) generation | `Qt_i`: token $t$ is at position $i$ | one token per position, `eos` then padding |
| (b) features | one per B-RASP vector and position | the B-RASP evaluation, as in `b_rasp.ipynb` |
| (c) transitions | `w{n}_{k}`, with weight $c$ | the step after position $n$ has probability $c$ |
| (d) property | `M{s}_{i}`: the DFA is in state $s$ after position $i$ | the DFA run, and acceptance |

Only the transition variables (c) carry weights. All other variables are **auxiliary** (weight 1 for both values), and they are fully determined by the tokens, so each generation path is counted exactly once.

We build each piece for the example `prompt, K = "ab", 3`, and print what it produces.

## (a) Generation

The string has $h = |u|$ prompt positions, then up to $K$ generated positions, then one more position for the final `eos`: $H + 1$ positions with $H = h + K$. Each position holds one of the tokens $a$, $b$, `eos` or `pad`. In the prompt, these are constants. After it, they are variables `Qt_i`, constrained so that:

- each position holds exactly one token (`Xor` with several arguments means *exactly one*),
- a position is `pad` exactly when an earlier one is `eos` or `pad` (so a stopped generation has a unique padding),
- the last position $H + 1$ holds no symbol of $\Sigma$, so at most $K$ symbols are generated, and `eos` is the only way to end.

In [13]:
TOKENS = ALPHABET + ["eos", "pad"]

def token_vectors(prompt, K):
    """Q[t][i] for tokens t and positions i = 1..H+1: a constant in the prompt, a variable after it."""
    h, H = len(prompt), len(prompt) + K
    return {t: Vector([(prompt[i - 1] == t) if i <= h else f"Q{t}_{i}" for i in range(1, H + 2)])
            for t in TOKENS}

def generation_constraints(Q, h, H):
    formulas = []
    for i in range(h + 1, H + 2):
        formulas.append(f"Xor({', '.join(Q[t][i] for t in TOKENS)})")
        stopped_before = Or(Q["eos"][i - 1], Q["pad"][i - 1]) if i > 1 else False
        formulas.append(Iff(Q["pad"][i], stopped_before))
    formulas += [Not(Q[a][H + 1]) for a in ALPHABET]
    return formulas

h, H = len(prompt), len(prompt) + K
Q = token_vectors(prompt, K)
print("tokens:", {t: Q[t].values for t in TOKENS})
print()
for f in generation_constraints(Q, h, H):
    print(f)

tokens: {'a': [True, False, 'Qa_3', 'Qa_4', 'Qa_5', 'Qa_6'], 'b': [False, True, 'Qb_3', 'Qb_4', 'Qb_5', 'Qb_6'], 'eos': [False, False, 'Qeos_3', 'Qeos_4', 'Qeos_5', 'Qeos_6'], 'pad': [False, False, 'Qpad_3', 'Qpad_4', 'Qpad_5', 'Qpad_6']}

Xor(Qa_3, Qb_3, Qeos_3, Qpad_3)
Not(Qpad_3)
Xor(Qa_4, Qb_4, Qeos_4, Qpad_4)
Biconditional(Qpad_4, Or(Qeos_3, Qpad_3))
Xor(Qa_5, Qb_5, Qeos_5, Qpad_5)
Biconditional(Qpad_5, Or(Qeos_4, Qpad_4))
Xor(Qa_6, Qb_6, Qeos_6, Qpad_6)
Biconditional(Qpad_6, Or(Qeos_5, Qpad_5))
Not(Qa_6)
Not(Qb_6)


## (b) Features

The B-RASP features are compiled exactly as in `b_rasp.ipynb`: each vector gets one variable per position, defined by a biconditional. Positions in the prompt are constants, so their values simplify away.

We compile **once**, for all positions $1..H$, rather than once per prefix: with strict-past attention, the value at position $i$ does not depend on later positions, so it is the same in every prefix that contains $i$. (At padded positions the values are meaningless, but they are never read.)

In [14]:
class Compile(Builder):
    """Builds a program over the token variables Q, for positions 1..H."""
    def __init__(self, Q, H):
        self.Q, self.positions = Q, range(1, H + 1)
        self.formulas, self.variables = [], []

    def define(self, name, exprs):
        values = []
        for i, e in zip(self.positions, exprs):
            if isinstance(e, bool):
                values.append(e)  ## a constant: no variable needed
            else:
                self.formulas.append(Iff(f"{name}_{i}", e))
                self.variables.append(f"{name}_{i}")
                values.append(f"{name}_{i}")
        return Vector(values)

compiled = Compile(Q, H)
phi = features(compiled)
for name, f in zip(["cur_a", "prev_a", "even"], phi):
    print(f"{name:7}", f.values)
print()
for f in compiled.formulas:
    print(f)

cur_a   [True, False, 'Qa_3', 'Qa_4', 'Qa_5', 'Qa_6']
prev_a  [False, True, False, 'prev_a_4', 'prev_a_5']
even    [False, True, False, True, False]

Biconditional(prev_a_4, Qa_3)
Biconditional(prev_a_5, Qa_4)


## (c) Transitions

At each step $n = h, \dots, H$ (after position $n$), the model draws the token at position $n+1$. The features at position $n$ select a row of the readout $r$. For each probability value $c$ that the step can have, a weighted variable `w{n}_{k}` is defined to be true exactly when the token drawn has probability $c$:

$$w_{n,k} \;\leftrightarrow\; \bigvee_{\substack{v \in \{0,1\}^d,\ t:\ r(v)(t) = c}} \Big(\Phi(n) = v\Big) \wedge \big(\text{token } t \text{ at } n+1\big)$$

and its literal weights are $w(w_{n,k}) = c$, $w(\neg w_{n,k}) = 1$. In `symprog`, this is what **categorical** variables do: a negated categorical variable has weight 1.

A step is *live* when position $n$ holds a symbol (always, in the prompt). A live step must take a transition with positive probability; then exactly one `w{n}_{k}` is true, and contributes $c$. After `eos`, all `w{n}_{k}` are false, and contribute 1. So the weight of an assignment is the probability of its path.

In [15]:
def transition_constraints(phi, Q, h, H):
    formulas, weights = [], {}
    for n in range(h, H + 1):
        ## the token drawn at step n has probability c in the cases cases[c]
        cases = {}
        if n == 0:
            for t, c in FIRST.items():
                if c > 0:
                    cases.setdefault(c, []).append(Q[t][1])
        else:
            for v in itertools.product([False, True], repeat=len(phi)):
                features_are_v = And(*[f[n] if bit else Not(f[n]) for f, bit in zip(phi, v)])
                for t, c in readout(v).items():
                    if c > 0:
                        cases.setdefault(c, []).append(And(features_are_v, Q[t][n + 1]))
        step = []
        for k, (c, terms) in enumerate(cases.items()):
            weights[f"w{n}_{k}"] = c
            formulas.append(Iff(f"w{n}_{k}", Or(*terms)))
            step.append(f"w{n}_{k}")
        ## a live step takes a transition with positive probability
        live = True if n <= h else Or(*[Q[a][n] for a in ALPHABET])
        formulas.append(Implies(live, Or(*step)))
    return formulas, weights

transitions, weights = transition_constraints(phi, Q, h, H)
print("weights:", weights)
print()
for f in transitions[:4]:
    print(f)
print("...")

weights: {'w2_0': 0.25, 'w2_1': 0.75, 'w2_2': 0.5, 'w3_0': 0.25, 'w3_1': 0.75, 'w3_2': 0.5, 'w4_0': 0.25, 'w4_1': 0.75, 'w4_2': 0.5, 'w5_0': 0.25, 'w5_1': 0.75, 'w5_2': 0.5}

Biconditional(w2_0, Qa_3)
Biconditional(w2_1, Qb_3)
Not(w2_2)
Or(w2_0, w2_1, w2_2)
...


## (d) Property

The DFA starts in the state reached after the prompt, and then follows the generated symbols. On `eos` and `pad` it stays where it is, so its state at position $H$ is its state after reading $uv$. One variable `M{s}_{i}` per state and position:

In [16]:
def monitor_constraints(Q, prompt, h, H):
    m = {s: s == dfa_state(prompt) for s in STATES}  ## state after the prompt: constants
    formulas, variables = [], []
    for i in range(h + 1, H + 1):
        symbol_at_i = Or(*[Q[a][i] for a in ALPHABET])
        new_m = {}
        for s2 in STATES:
            step = Or(*[And(m[s], Q[a][i]) for s in STATES for a in ALPHABET if DELTA[s, a] == s2])
            stay = And(Not(symbol_at_i), m[s2])
            formulas.append(Iff(f"M{s2}_{i}", Or(step, stay)))
            variables.append(f"M{s2}_{i}")
            new_m[s2] = f"M{s2}_{i}"
        m = new_m
    accept = Or(*[m[s] for s in ACCEPTING])
    return formulas, variables, accept

monitor, monitor_variables, accept = monitor_constraints(Q, prompt, h, H)
for f in monitor[:3]:
    print(f)
print("...")
print("accept:", accept)

Biconditional(M0_3, Qa_3)
Biconditional(M1_3, Not(Or(Qa_3, Qb_3)))
Biconditional(M2_3, Qb_3)
...
accept: Or(M0_5, M1_5)


## Counting

All the formulas go into one solver, together with `accept`. The token, feature and monitor variables are auxiliary, and the transition variables are weighted categoricals. With no query, `Prob` is the weighted model count, i.e., $\Pr^K_A(L_u \mid u)$:

In [17]:
token_variables = [Q[t][i] for t in TOKENS for i in range(h + 1, H + 2)]
aux = token_variables + compiled.variables + monitor_variables

formulas = generation_constraints(Q, h, H) + compiled.formulas + transitions + monitor + [accept]
formulas = [f for f in formulas if f is not True]
print(len(formulas), "formulas,", len(weights), "weighted variables,", len(aux), "auxiliary variables")

solver = Solver.from_kwargs(
    var_list=list(weights) + aux,
    var_weights=weights,
    aux_vars=set(aux),
    categoricals=set(weights),  ## w(not w_nk) = 1
)
solver.add(formulas)

print(f"weighted model count: {solver.query(mode='Prob').inference.item():.6f}")
print(f"brute force:          {float(brute_force(prompt, K)):.6f}  (= {brute_force(prompt, K)})")

38 formulas, 12 weighted variables, 27 auxiliary variables
weighted model count: 0.105469
brute force:          0.105469  (= 27/256)


## Checking against brute force

The same steps, for several prompts (including the empty one) and caps. If some formula simplifies to `False` (e.g., the property is unreachable), the probability is 0.

In [18]:
def encode(prompt, K):
    """Pieces (a)-(d) above, for one prompt and cap."""
    h, H = len(prompt), len(prompt) + K
    Q = token_vectors(prompt, K)
    compiled = Compile(Q, H)
    phi = features(compiled)
    transitions, weights = transition_constraints(phi, Q, h, H)
    monitor, monitor_variables, accept = monitor_constraints(Q, prompt, h, H)
    formulas = generation_constraints(Q, h, H) + compiled.formulas + transitions + monitor + [accept]
    aux = [Q[t][i] for t in TOKENS for i in range(h + 1, H + 2)] + compiled.variables + monitor_variables
    return [f for f in formulas if f is not True], weights, aux

def solver_for(formulas, weights, aux):
    solver = Solver.from_kwargs(var_list=list(weights) + aux, var_weights=weights,
                                aux_vars=set(aux), categoricals=set(weights))
    solver.add(formulas)
    return solver

print("prompt  " + "  ".join(f"K={K:<6}" for K in range(6)))
for prompt in ["", "a", "ab", "bab"]:
    row = []
    for K in range(6):
        formulas, weights, aux = encode(prompt, K)
        wmc = 0.0 if False in formulas else solver_for(formulas, weights, aux).query(mode="Prob").inference.item()
        assert abs(wmc - float(brute_force(prompt, K))) < 1e-6
        row.append(f"{wmc:.4f}")
    print(f"{prompt!r:7} " + "  ".join(f"{p:8}" for p in row))
print("all equal to brute force")

prompt  K=0       K=1       K=2       K=3       K=4       K=5     


''      0.0000    0.0000    0.2500    0.2812    0.3398    0.3472  


'a'     0.0000    0.3750    0.4219    0.5156    0.5273    0.5508  


'ab'    0.0000    0.0000    0.0938    0.1055    0.1289    0.1318  


'bab'   0.0000    0.1250    0.1406    0.1641    0.1670    0.1729  
all equal to brute force


**PROB and THRESH.** With this number, the two decision problems of the draft are a comparison: PROB asks whether $\Pr^K_A(L_u \mid u) = \lambda$, and THRESH whether $\Pr^K_A(L_u \mid u) \ge \lambda$. (Floating-point weights are exact here, since all probabilities are dyadic.)

# 5. What else the circuit gives us

The compiled circuit represents all accepted generation paths with their probabilities, so other questions are queries on the same solver.

#### The most likely accepted continuation

The assignment with the highest weight is the most likely path that stops within the cap and satisfies the property:

In [19]:
prompt, K = "ab", 6
formulas, weights, aux = encode(prompt, K)
solver = solver_for(formulas, weights, aux)

best = solver.query(mode="ViterbiAssignment").variables
h, H = len(prompt), len(prompt) + K
continuation = "".join(a for i in range(h + 1, H + 1) for a in ALPHABET if best.get(f"Q{a}_{i}"))
print(f"most likely accepted continuation of {prompt!r}: {continuation!r},",
      f"probability {path_probability(prompt, continuation)}")

## brute force: the most likely accepted continuation
candidates = ["".join(v) for m in range(K + 1) for v in itertools.product(ALPHABET, repeat=m)
              if dfa_state(prompt + "".join(v)) in ACCEPTING]
assert path_probability(prompt, continuation) == max(path_probability(prompt, v) for v in candidates)

most likely accepted continuation of 'ab': 'aa', probability 3/32


#### Conditional probabilities

Given that the output is accepted, what is the distribution of each generated position? A query divides by the total weight, so `query=...` gives a conditional probability:

In [20]:
print(f"Pr(property holds) = {solver.query(mode='Prob').inference.item():.4f}\n")
for i in range(h + 1, H + 2):
    dist = {t: solver.query(query=f"Q{t}_{i}").inference.item() for t in TOKENS}
    print(f"position {i}: " + "  ".join(f"{t}={p:.3f}" for t, p in dist.items()))

Pr(property holds) = 0.1377

position 3: a=1.000  b=0.000  eos=0.000  pad=0.000


position 4: a=0.941  b=0.059  eos=0.000  pad=0.000
position 5: a=0.263  b=0.056  eos=0.681  pad=0.000


position 6: a=0.223  b=0.011  eos=0.085  pad=0.681
position 7: a=0.053  b=0.011  eos=0.170  pad=0.766


position 8: a=0.043  b=0.000  eos=0.021  pad=0.936
position 9: a=0.000  b=0.000  eos=0.043  pad=0.957


#### Size

The number of formulas grows linearly with $K$ here (the attention in `prev_a` simplifies to the previous symbol; an attention with a nontrivial score has formulas of size $O(i)$ at position $i$). The circuit, and the compile time, grow faster:

In [21]:
print(" K  formulas  Pr(no bb | 'ab')  time")
for K in [5, 10, 15]:
    start = time.perf_counter()
    formulas, weights, aux = encode("ab", K)
    prob = solver_for(formulas, weights, aux).query(mode="Prob").inference.item()
    print(f"{K:2d}  {len(formulas):8d}  {prob:16.6f}  {time.perf_counter() - start:.2f}s")

 K  formulas  Pr(no bb | 'ab')  time


 5        58          0.131836  0.08s


10       108          0.140442  0.45s


15       158          0.140616  1.88s


# Notes

- **Readouts with few distinct probabilities are cheaper:** each step has one weighted variable per probability value, not per (features, symbol) pair.
- **The readout is a table over $2^d$ feature vectors**, so this works for a handful of features. A transformer translated into B-RASP (Section 8 of the draft) has many more, and would need $r$ in a compact form.
- **Unbounded generation** ($K = \infty$) is not a weighted model count of a finite formula: it needs Markov-chain methods on the (exponentially large) state space.